## 0. Import & Setup

In [1]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score
)

from src.preprocessing import (
    LANGUAGES,
    NUM_LABELS,
    LABEL_REMAP_INVERSE,
    preprocess_dataset,
    get_splits_as_df,
    compute_class_weights
)

import warnings
warnings.filterwarnings('ignore')

c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Load & Preprocess Bahasa

In [2]:
datasets_per_lang = {}

for lang in LANGUAGES:
    raw = load_dataset(
        'indonlp/nusatranslation_senti',
        name=f'nusatranslation_senti_{lang}_nusantara_text',
        trust_remote_code=True
    )
    datasets_per_lang[lang] = preprocess_dataset(raw)
    print(f"[{lang}] preprocessed.")

Map: 100%|██████████| 1200/1200 [00:00<00:00, 24859.31 examples/s]


[jav] preprocessed.


Map: 100%|██████████| 1200/1200 [00:00<00:00, 26979.81 examples/s]


[min] preprocessed.


Map: 100%|██████████| 1200/1200 [00:00<00:00, 26508.34 examples/s]

[sun] preprocessed.


In [3]:
results = []

for lang in LANGUAGES:
    splits  = get_splits_as_df(datasets_per_lang[lang])
    
    X_train = splits['train']['text']
    y_train = splits['train']['label']
    X_val   = splits['validation']['text']
    y_val   = splits['validation']['label']
    X_test  = splits['test']['text']
    y_test  = splits['test']['label']

    # TF-IDF vectorizer: unigram + bigram, max 10k features
    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        max_features=10000,
        sublinear_tf=True
    )
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_val_tfidf   = vectorizer.transform(X_val)
    X_test_tfidf  = vectorizer.transform(X_test)

    # Class weights
    cw = compute_class_weights(y_train.tolist())

    # Model definitions
    models = {
        'LogisticRegression': LogisticRegression(
            class_weight=cw,
            max_iter=1000,
            random_state=42
        ),
        'LinearSVC': LinearSVC(
            class_weight=cw,
            max_iter=2000,
            random_state=42
        )
    }

    for model_name, model in models.items():
        model.fit(X_train_tfidf, y_train)
        
        # Evaluasi di validation dan test
        for split_name, X_vec, y_true in [
            ('validation', X_val_tfidf, y_val),
            ('test', X_test_tfidf, y_test)
        ]:
            y_pred   = model.predict(X_vec)
            macro_f1 = f1_score(y_true, y_pred, average='macro')
            
            results.append({
                'language'  : lang,
                'model'     : model_name,
                'split'     : split_name,
                'macro_f1'  : round(macro_f1, 4)
            })
        
        print(f"[{lang}] {model_name} done.")

results_df = pd.DataFrame(results)
results_df.to_csv('../results/classical_results.csv', index=False)
print("\nSaved to results/classical_results.csv")

[jav] LogisticRegression done.
[jav] LinearSVC done.
[min] LogisticRegression done.
[min] LinearSVC done.
[sun] LogisticRegression done.
[sun] LinearSVC done.

Saved to results/classical_results.csv


## 3. Result (Classical)

In [4]:
# Pivot supaya mudah dibaca
pivot = results_df[results_df['split'] == 'test'].pivot(
    index='language',
    columns='model',
    values='macro_f1'
)

print("=== Classical Baseline -- Test Set Macro F1 ===")
print(pivot.to_string())

=== Classical Baseline -- Test Set Macro F1 ===
model     LinearSVC  LogisticRegression
language                               
jav          0.8137              0.7868
min          0.8265              0.8097
sun          0.8228              0.7935
